<a href="https://colab.research.google.com/github/claudia-z712/HW_3_Submission/blob/main/Zam_C_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1>Zambrano_Claudia_HW3</h1></center>
<br>
<br>

Name: Claudia Zambrano
<br>
Github Username: claudia-z712
<br>
USC ID: 8224948861

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [ ]:
import os
import zipfile
import shutil

Get the AReM Data Set

In [11]:
# Extract the uploaded ZIP file
zip_filepath = "/content/AReM.zip"
extract_path = "AReM_data"

print("Extracting the local dataset...")
with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print(f"Extraction complete to: {extract_path}")

# List contents of the extracted folder
if os.path.exists(extract_path):
    print("Contents:", os.listdir(extract_path))

Extracting the local dataset...
Extraction complete to: AReM_data
Contents: ['AReM']


### (b) Test and Train Data

In [12]:
import os
import shutil

# Source and target directories
src_dir = "AReM_data"
# Note: Sometimes extracted contents are nested in a subdirectory (like 'AReM' or similar)
# Let's locate the actual root folder containing the class directories
subfolders = [f for f in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, f))]
actual_src = os.path.join(src_dir, subfolders[0]) if len(subfolders) == 1 and subfolders[0].lower() in ['arem', 'data'] else src_dir

test_dir = "Testing-Data"
train_dir = "Training-Data"

os.makedirs(test_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)

# Iterate through each folder (category) inside the source dataset
for category in os.listdir(actual_src):
    category_path = os.path.join(actual_src, category)
    if not os.path.isdir(category_path) or category.startswith('.'):
        continue

    # Create category subdirectories inside Testing and Training folders
    os.makedirs(os.path.join(test_dir, category), exist_ok=True)
    os.makedirs(os.path.join(train_dir, category), exist_ok=True)

    # Get and sort all dataset files (assuming format like dataset1.csv, dataset2.csv, etc.)
    files = sorted([f for f in os.listdir(category_path) if f.endswith('.csv')])

    for f in files:
        src_file_path = os.path.join(category_path, f)

        # Determine if the file is a test or train instance
        is_test = False

        if category in ['bending1', 'bending2']:
            # Place dataset1 and dataset2 into testing
            if f in ['dataset1.csv', 'dataset2.csv']:
                is_test = True
        else:
            # Place dataset1, dataset2, and dataset3 into testing
            if f in ['dataset1.csv', 'dataset2.csv', 'dataset3.csv']:
                is_test = True

        # Copy to the corresponding target directory
        if is_test:
            shutil.copy(src_file_path, os.path.join(test_dir, category, f))
        else:
            shutil.copy(src_file_path, os.path.join(train_dir, category, f))

print("Splitting complete!")
print(f"Testing-Data classes: {os.listdir(test_dir)}")
print(f"Training-Data classes: {os.listdir(train_dir)}")

Splitting complete!
Testing-Data classes: ['cycling', 'lying', 'bending2', 'sitting', 'standing', 'bending1', 'walking']
Training-Data classes: ['cycling', 'lying', 'bending2', 'sitting', 'standing', 'bending1', 'walking']


In [18]:
import shutil
import os

# Replacing broken dataset with fixed dataset
source_clean = "/content/CLEANdataset4.csv"
target_original = "/content/AReM_data/AReM/bending2/dataset4.csv"
target_train = "/content/Training-Data/bending2/dataset4.csv"

# Perform the replacement
if os.path.exists(source_clean):
    # Replace in original raw dataset
    shutil.copy(source_clean, target_original)
    print(f"Successfully replaced original file at: {target_original}")

    # Replace in training data directory if it exists
    if os.path.exists(target_train):
        shutil.copy(source_clean, target_train)
        print(f"Successfully replaced split training file at: {target_train}")
else:
    print(f"Error: {source_clean} was not found in the current directory.")

Successfully replaced original file at: /content/AReM_data/AReM/bending2/dataset4.csv
Successfully replaced split training file at: /content/Training-Data/bending2/dataset4.csv


### (c) Feature Extraction

#### i. Research

In time series classification, the usual time-domain feastures are mean, mediam, mode, standard deviation, IQR, Root Mean Squared, Skewedness, and Kurtosis.

#### ii. Extraction

In [21]:
import os
import pandas as pd
import numpy as np

src_dir = "/content/AReM_data/AReM"
# Get all subdirectories (categories) in alphabetical order
categories = sorted([d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))])

all_instances_features = []

for category in categories:
    category_path = os.path.join(src_dir, category)
    files = sorted(
        [f for f in os.listdir(category_path) if f.endswith('.csv')],
        key=lambda x: int(''.join(filter(str.isdigit, x)) or 0)
    )

    for f in files:
        file_path = os.path.join(category_path, f)

        # Standardized reading mechanism:
        # Skip first 4 rows (0-indexed 0,1,2,3), so Row 5 (index 4) becomes the header
        try:
            df = pd.read_csv(file_path, skiprows=4)
        except Exception:
            df = pd.read_csv(file_path, skiprows=4, sep=r'\s+')

        # Drop fully empty rows/columns
        df = df.dropna(how='all', axis=0).dropna(how='all', axis=1)

        # Handle instances where the entire row was parsed as a single comma-separated string
        if df.shape[1] == 1:
            col_name = df.columns[0]
            df_split = df.iloc[:, 0].astype(str).str.split(',', expand=True)
            header_split = str(col_name).split(',')
            if len(header_split) == df_split.shape[1]:
                df_split.columns = header_split
            df = df_split

        # Clean any trailing commas or whitespaces from column names
        df.columns = [str(c).strip().replace('#', '').strip() for c in df.columns]

        # Convert everything to numeric values
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]]) # Drop rows where time is NaN

        # The first column is 'time', the next 6 are the sensor time series
        time_col = df.columns[0]
        ts_cols = df.columns[1:7]

        ts_data = df[ts_cols]

        instance_info = {
            'Category': category,
            'Instance': f.replace('.csv', '')
        }

        # Extract features for each of the 6 columns
        for col_idx, col_name in enumerate(ts_cols):
            col_data = ts_data[col_name]

            # Calculate time-domain statistics and round to 3 decimal places
            instance_info[f'col{col_idx+1}_min'] = round(float(col_data.min()), 3)
            instance_info[f'col{col_idx+1}_max'] = round(float(col_data.max()), 3)
            instance_info[f'col{col_idx+1}_mean'] = round(float(col_data.mean()), 3)
            instance_info[f'col{col_idx+1}_median'] = round(float(col_data.median()), 3)
            instance_info[f'col{col_idx+1}_std'] = round(float(col_data.std()), 3)
            instance_info[f'col{col_idx+1}_1st_quart'] = round(float(col_data.quantile(0.25)), 3)
            instance_info[f'col{col_idx+1}_3rd_quart'] = round(float(col_data.quantile(0.75)), 3)

        all_instances_features.append(instance_info)

# Create final structured table
features_df = pd.DataFrame(all_instances_features)
print(f"Extracted features for {features_df.shape[0]} instances across all categories.")
pd.set_option('display.max_columns', None)
display(features_df)

Extracted features for 88 instances across all categories.


,Category,Instance,col1_min,col1_max,col1_mean,col1_median,col1_std,col1_1st_quart,col1_3rd_quart,col2_min,col2_max,col2_mean,col2_median,col2_std,col2_1st_quart,col2_3rd_quart,col3_min,col3_max,col3_mean,col3_median,col3_std,col3_1st_quart,col3_3rd_quart,col4_min,col4_max,col4_mean,col4_median,col4_std,col4_1st_quart,col4_3rd_quart,col5_min,col5_max,col5_mean,col5_median,col5_std,col5_1st_quart,col5_3rd_quart,col6_min,col6_max,col6_mean,col6_median,col6_std,col6_1st_quart,col6_3rd_quart
0,bending1,dataset1,37.25,45.00,40.625,40.50,1.477,39.25,42.000,0.0,1.30,0.359,0.430,0.323,0.00,0.500,4.00,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.000,1.120,27.25,38.25,34.311,35.00,2.188,33.000,36.00,0.00,1.92,0.571,0.43,0.583,0.000,1.300
1,bending1,dataset2,38.00,45.67,42.813,42.50,1.436,42.00,43.670,0.0,1.22,0.372,0.470,0.289,0.00,0.500,2.00,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.000,1.145,27.67,38.50,33.025,33.00,1.995,32.000,34.50,0.00,3.11,0.571,0.43,0.601,0.000,1.300
2,bending1,dataset3,35.00,47.40,43.955,44.33,1.559,43.00,45.000,0.0,1.70,0.426,0.470,0.339,0.00,0.500,6.50,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.000,0.830,29.00,38.50,35.588,36.00,2.000,35.362,36.50,0.00,1.79,0.493,0.43,0.514,0.000,0.940
3,bending1,dataset4,33.00,47.75,42.180,43.50,3.671,39.15,45.000,0.0,3.00,0.696,0.500,0.631,0.00,1.120,8.50,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.430,1.300,20.00,38.67,33.494,35.00,3.849,30.457,36.33,0.00,2.18,0.614,0.50,0.524,0.000,1.000
4,bending1,dataset5,33.00,45.75,41.678,41.75,2.243,41.33,42.750,0.0,2.83,0.536,0.500,0.405,0.43,0.710,3.00,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.430,1.120,23.67,37.50,29.857,30.00,2.411,28.457,31.25,0.00,1.79,0.383,0.43,0.389,0.000,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking,dataset11,19.50,45.33,33.587,34.25,4.651,30.25,37.000,0.0,14.67,4.577,4.260,2.461,2.87,6.105,7.33,23.25,15.318,15.330,2.970,13.25,17.270,0.0,9.00,3.227,3.10,1.647,2.050,4.260,6.33,25.00,15.892,16.00,3.284,13.730,18.25,0.00,8.32,3.260,3.11,1.640,2.050,4.322
84,walking,dataset12,19.75,45.50,34.323,35.25,4.752,31.00,38.000,0.0,13.47,4.456,3.900,2.599,2.49,5.970,6.25,22.25,15.210,15.250,2.711,13.50,17.000,0.0,9.00,3.277,3.11,1.500,2.170,4.390,6.25,23.25,15.550,15.75,3.120,13.500,17.75,0.00,9.67,3.433,3.20,1.733,2.157,4.565
85,walking,dataset13,19.50,46.00,34.546,35.25,4.842,31.25,37.812,0.0,12.47,4.372,4.135,2.360,2.49,5.800,7.00,22.67,15.186,15.250,2.758,13.25,17.000,0.0,8.34,3.029,2.86,1.665,1.790,4.110,7.00,22.75,15.760,15.75,2.823,14.000,17.75,0.00,10.00,3.338,3.08,1.657,2.160,4.335
86,walking,dataset14,23.50,46.25,34.873,35.25,4.532,31.75,38.250,0.0,14.82,4.381,3.925,2.435,2.50,5.933,6.67,24.25,15.471,15.500,2.813,13.50,17.250,0.0,9.90,3.206,3.03,1.519,2.058,4.190,5.50,23.50,15.874,16.00,3.131,13.750,18.00,0.00,9.51,3.425,3.27,1.691,2.170,4.500


#### iii. Standard Deviation

In [22]:
import pandas as pd

# Exclude 'Category' and 'Instance' columns to focus only on numerical feature columns
feature_cols = [col for col in features_df.columns if col not in ['Category', 'Instance']]

# Calculate the standard deviation for each feature
features_std = features_df[feature_cols].std()

# Format as a clean DataFrame for display
features_std_df = pd.DataFrame(features_std, columns=['Standard Deviation'])
display(features_std_df)



,Standard Deviation
col1_min,9.569975
col1_max,4.394362
col1_mean,5.335771
col1_median,5.440054
col1_std,1.772188
col1_1st_quart,6.153548
col1_3rd_quart,5.138944
col2_min,0.000000
col2_max,5.062729
col2_mean,1.574190


In [23]:
import numpy as np
import pandas as pd

# Exclude non-numeric target columns
feature_cols = [col for col in features_df.columns if col not in ['Category', 'Instance']]

# Settings for bootstrapping
num_bootstrap_samples = 1000
confidence_level = 0.90
lower_percentile = (1 - confidence_level) / 2 * 100
upper_percentile = (1 + confidence_level) / 2 * 100

bootstrap_results = []

# Seed for reproducibility
np.random.seed(42)

for col in feature_cols:
    values = features_df[col].dropna().values
    n = len(values)

    # Generate bootstrap samples and calculate the standard deviation for each sample
    bootstrap_samples = np.random.choice(values, size=(num_bootstrap_samples, n), replace=True)
    bootstrap_stds = np.std(bootstrap_samples, axis=1, ddof=1)

    # Calculate lower and upper percentiles for the confidence interval
    ci_lower = np.percentile(bootstrap_stds, lower_percentile)
    ci_upper = np.percentile(bootstrap_stds, upper_percentile)

    bootstrap_results.append({
        'Feature': col,
        'Observed Std': np.std(values, ddof=1),
        '90% CI Lower': ci_lower,
        '90% CI Upper': ci_upper
    })

# Convert to DataFrame for easier viewing
bootstrap_ci_df = pd.DataFrame(bootstrap_results)
display(bootstrap_ci_df)

,Feature,Observed Std,90% CI Lower,90% CI Upper
0,col1_min,9.569975,8.267407,10.692667
1,col1_max,4.394362,3.370173,5.292857
2,col1_mean,5.335771,4.720049,5.902023
3,col1_median,5.440054,4.763061,6.017586
4,col1_std,1.772188,1.571170,1.953265
5,col1_1st_quart,6.153548,5.545704,6.660430
6,col1_3rd_quart,5.138944,4.387648,5.841510
7,col2_min,0.000000,0.000000,0.000000
8,col2_max,5.062729,4.627246,5.404118
9,col2_mean,1.574190,1.401804,1.714106


#### iv. Select Features

I would consider mean, maximum, and minimum to be the most important features. Mean can give us a baseline of what should be expected while maximum and minimum gives us the expected range of data points to be expected and gives us the starting point for the rest of our calculations.

Another feature to consider would be standard deviation. It defines how the signal fluctuates around the mean, given we will eventually try to define a movement based on signals, its important to understand the variability within each.

Median would be another feature to keep notice. While the minumum and maximum can be useful, median would be less affected if there were any outstanding outliers.






## 2. ISLR 3.7.4

### (a) Linear Train

Training RSS for cubic regression should be smaller than linear regression because cubic regression should fit more of the data in our set.

### (b) Linear Test

Test RSS for linear regression should be smaller than cubic because the true relationship should be linear between X and Y.

### (c) Not Linear Train

If the true relationship between X and Y is not linear then we could consider cubic regression since it poses to be a more flexible model.

### (d) Not Linear Testing

There is not enough information available to determine which RSS would be smaller since we do not know the true relationship between X and Y.